In [ ]:
import sys
sys.argv = [sys.argv[0]]

In [ ]:
import os
import uuid
import shutil
import asyncio
import tempfile
import aiohttp
import fitz
import nest_asyncio
import numpy as np
from dotenv import load_dotenv
from flask import Flask, request, jsonify
from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status
from google import genai
from google.genai import types
from groq import Groq
import httpx
from openai import OpenAI

In [ ]:
nest_asyncio.apply()

In [ ]:
app = Flask(__name__)

VALID_AUTH_KEY = os.getenv("VALID_AUTH_KEY")

class RAGSystem:
    def __init__(self, working_dir="./trial_light_rag", embedding_model_name="all-MiniLM-L6-v2"):
        self.working_dir = working_dir
        self.embedding_model_name = embedding_model_name
        self.embedding_model = None
        self.rag = None
        self.gemini_api_key = os.getenv("GEMINI_API_KEY")
        self.groq_api_key = os.getenv("GROQ_API_KEY")

    def _setup_working_dir(self):
        if os.path.exists(self.working_dir):
            shutil.rmtree(self.working_dir)
        os.mkdir(self.working_dir)
        return self.working_dir

    async def _llm_model_func(self, prompt, system_prompt=None, history_messages=None, **kwargs):
        if history_messages is None:
            history_messages = []
    
        client = OpenAI(
            api_key="abc-123",
            base_url="http://localhost:3000/v1",
        )
    
        # Initialize messages list
        messages = []
    
        # Add system prompt if provided
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
    
        # Add history messages
        for msg in history_messages:
            messages.append({"role": msg['role'], "content": msg['content']})
    
        # Add the current user prompt
        messages.append({"role": "user", "content": prompt})
    
        response = client.chat.completions.create(
            model="Qwen/Qwen3-30B-A3B-Instruct-2507",
            messages=messages,
            temperature=0.1,
        )
    
        return response.choices[0].message.content

    async def _embedding_func(self, texts):
        if self.embedding_model is None:
            self.embedding_model = SentenceTransformer(self.embedding_model_name, trust_remote_code=True, device="cuda")
        return self.embedding_model.encode(texts, batch_size=32, convert_to_numpy=True, device="cuda")

    async def initialize(self):
        try:
            self.rag = LightRAG(
                working_dir=self.working_dir,
                llm_model_func=self._llm_model_func,
                embedding_func=EmbeddingFunc(
                    embedding_dim=384,
                    max_token_size=8192,
                    func=self._embedding_func,
                ),
                max_parallel_insert=4,
                chunk_token_size=8000, 
                chunk_overlap_token_size=512,
            )
            await self.rag.initialize_storages()
            await initialize_pipeline_status()
            return True
        except Exception as e:
            print(f"Initialization error: {e}")
            return False

    async def download_pdf(self, url, temp_file_path):
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url) as response:
                    if response.status != 200:
                        print(f"Failed to download PDF: HTTP {response.status}")
                        return False
                    with open(temp_file_path, "wb") as f:
                        f.write(await response.read())
            return True
        except Exception as e:
            print(f"Download error: {e}")
            return False

    def pdf_to_text(self, pdf_path, txt_output_path):
        try:
            doc = fitz.open(pdf_path)
            with open(txt_output_path, "w", encoding="utf-8") as out_file:
                for page in doc:
                    text = page.get_text()
                    out_file.write(text + "\n")
            doc.close()
            return True
        except Exception as e:
            print(f"PDF conversion error: {e}")
            return False

    async def ingest_pdf_from_url(self, pdf_url):
        try:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
                temp_pdf_path = temp_pdf.name
                if not await self.download_pdf(str(pdf_url), temp_pdf_path):
                    return False

                txt_output_path = os.path.join(self.working_dir, f"temp_{uuid.uuid4()}.txt")
                if not self.pdf_to_text(temp_pdf_path, txt_output_path):
                    return False

                with open(txt_output_path, "r", encoding="utf-8") as file:
                    text = file.read()
                await self.rag.ainsert(text)

                os.remove(txt_output_path)
                return True
        except Exception as e:
            print(f"Ingestion error: {e}")
            return False
        finally:
            if os.path.exists(temp_pdf_path):
                os.remove(temp_pdf_path)

    async def ingest_from_txt(self, txt_path):
        try:
            with open(txt_path, "r", encoding="utf-8") as file:
                    text = file.read()
            await self.rag.ainsert(text)
            return True
        except Exception as e:
            print(f"Ingestion error: {e}")
            return False
        

    async def query(self, question):
        if not self.rag:
            raise ValueError("RAG system not initialized.")

        try:
            res = await self.rag.aquery(
                question,
                param=QueryParam(mode="hybrid", enable_rerank=False)
            )

            prompt = f"""
            Based on the provided context below, please provide a shorter, crisp and accurate response having all important information required by the question: "{question}".
            \nContext: {res}
            \nDo not mention anything about references in the final short answer. Do not use new line for answers.
            \nIf the context is empty or not found then use your intelligence and answer the question accordingly with a shorter, crisp and accurate response.
            \nAnswer:"""

            groq_client = OpenAI(
                    api_key="abc-123",
                    base_url="http://localhost:3000/v1",
                )

            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {"role": "user", "content": prompt},
                    {"role": "system", "content": "You are a helpful assistant helping in summarizing long answers into short, crisp and to the point answers, not missing any important points in the answer."}
                ],
                model="Qwen/Qwen3-30B-A3B-Instruct-2507",
            )

            return chat_completion.choices[0].message.content
        except Exception as e:
            print(f"Query error: {e}")
            return ""

    async def process_questions(self, questions):
        answers = []
        for question in questions:
            answer = await self.query(question)
            answers.append(answer)
        return answers

    async def finalize(self):
        if self.rag:
            await self.rag.finalize_storages()
            self.rag = None


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "message": "API is healthy"}), 200

@app.route("/hackrx/run", methods=["POST"])
def run_ingest_rag():
    auth_header = request.headers.get("Authorization", "")
    if not auth_header.startswith("Bearer ") or auth_header.split(" ")[1] != VALID_AUTH_KEY:
        return jsonify({"error": "Invalid or missing API key"}), 401

    data = request.get_json()
    pdf_url = data.get("documents")
    questions = data.get("questions")

    rag_system = RAGSystem(working_dir=f"./trial_light_rag_{uuid.uuid4()}")
    rag_system._setup_working_dir()

    async def process():
        if not await rag_system.initialize():
            return {"error": "Failed to initialize RAG system"}, 500
        if not await rag_system.ingest_pdf_from_url(pdf_url):
            return {"error": "Failed to process PDF"}, 500
        answers = await rag_system.process_questions(questions)
        await rag_system.finalize()
        return {"answers": answers}, 200

    return asyncio.run(process())


@app.route("/hackrx/check", methods=["POST"])
def check_rag():
    auth_header = request.headers.get("Authorization", "")
    if not auth_header.startswith("Bearer ") or auth_header.split(" ")[1] != VALID_AUTH_KEY:
        return jsonify({"error": "Invalid or missing API key"}), 401

    data = request.get_json()
    questions = data.get("questions")

    rag_system = RAGSystem(working_dir="./trial_light_rag_288842d7-087b-4484-978d-0b0e1f31810b")

    async def process():
        if not await rag_system.initialize():
            return {"error": "Failed to initialize RAG system"}, 500
        answers = await rag_system.process_questions(questions)
        return {"answers": answers}, 200

    return asyncio.run(process())


In [ ]:
app.run(host="0.0.0.0", port=8000)